# Bronze data profile (A4)

This notebook profiles the complete 2023 Bronze snapshot before Silver modelling.

| Scope | Result |

|---|---:|

| Records | 1,079,929 |

| Source fields | 133 |

| Schema baseline | `2023-a1` |

| Exact duplicate rows | 0 |

| Duplicate candidate keys | 0 |

All source fields remain strings in Bronze. Empty and whitespace-only strings are
counted as missing. Distinct counts and quality findings are exact. Raw minimum and
maximum values are lexical; dates and weights are parsed separately for meaningful
ranges. The complete machine-readable results are stored in
`docs/evidence/a4-profile.json`.


## 1. Load the completed Bronze snapshot


In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent

spark = (
    SparkSession.builder
    .appName("a4-profile-bronze")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

expected = json.loads(
    (ROOT / "src/bronze/expected_schema.json").read_text(encoding="utf-8")
)
source_columns = [field["name"] for field in expected["fields"]]

bronze_root = ROOT / "data/dev/bronze/fiskeridir_catch/period=2023"
snapshots = sorted(
    path for path in bronze_root.glob("source_version=*/records")
    if (path / "_SUCCESS").is_file()
)
if len(snapshots) != 1:
    raise RuntimeError(f"Expected one completed 2023 snapshot, found {len(snapshots)}")

snapshot = snapshots[0]

def q(column):
    """Reference a source field literally, including names containing dots."""
    return F.col(f"`{column.replace('`', '``')}`")

df = spark.read.parquet(str(snapshot)).select(*[q(c) for c in source_columns])
row_count = df.count()

print(f"Snapshot: {snapshot.relative_to(ROOT)}")
print(f"Rows: {row_count:,}; source fields: {len(source_columns)}")


26/11/08 19:02:06 WARN Utils: Your hostname, moon resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/08 19:02:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 19:02:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Snapshot: data/dev/bronze/fiskeridir_catch/period=2023/source_version=71783c32df9bdf3af86c0fa3ae4b56523d7dea4836349189490d2dcbf99a0592/records
Rows: 1,079,929; source fields: 133


## 2. Per-field completeness, cardinality and raw range


In [2]:
# Small batches bound Spark's aggregation state and use Parquet column pruning.
profile_values = {}
batch_size = 8

for start in range(0, len(source_columns), batch_size):
    expressions = []
    for index, column in enumerate(source_columns[start:start + batch_size], start=start):
        expressions.extend([
            F.sum(
                F.when(q(column).isNull() | (F.trim(q(column)) == ""), 1)
                 .otherwise(0)
            ).alias(f"m{index}"),
            F.countDistinct(q(column)).alias(f"d{index}"),
            F.min(q(column)).alias(f"lo{index}"),
            F.max(q(column)).alias(f"hi{index}"),
        ])
    profile_values.update(df.agg(*expressions).first().asDict())

field_profile = []
for index, column in enumerate(source_columns):
    missing = int(profile_values[f"m{index}"])
    field_profile.append({
        "column": column,
        "missing_count": missing,
        "missing_percent": round(100 * missing / row_count, 4),
        "distinct_raw_values": int(profile_values[f"d{index}"]),
        "raw_min": profile_values[f"lo{index}"],
        "raw_max": profile_values[f"hi{index}"],
    })

profile_df = spark.createDataFrame(field_profile)
profile_df.orderBy(F.desc("missing_count")).show(20, truncate=False)


26/11/08 19:02:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------------------+-------------------+-------------+---------------+---------------+-------+
|column                       |distinct_raw_values|missing_count|missing_percent|raw_max        |raw_min|
+-----------------------------+-------------------+-------------+---------------+---------------+-------+
|Mottaker ID                  |1                  |1079929      |100.0          |               |       |
|Neste mottaksstasjon         |42                 |1077137      |99.7415        |URY            |       |
|Forrige mottakstasjon        |46                 |1075921      |99.6289        |URY            |       |
|Mottakende fartøy rkal       |60                 |1075470      |99.5871        |VL1A           |       |
|Mottakende fartøy reg.merke  |60                 |1075463      |99.5865        |Ø 0300H        |       |
|Mottakende fartøytype (kode) |4                  |1075463      |99.5865        |05             |       |
|Mottakende fart.type         |4              

### Completeness findings

| Field | Missing rows | Missing % | Interpretation |

|---|---:|---:|---|

| `Mottaker ID` | 1,079,929 | 100.0000 | Unusable in this snapshot |

| `Neste mottaksstasjon` | 1,077,137 | 99.7415 | Optional movement field |

| `Forrige mottakstasjon` | 1,075,921 | 99.6289 | Optional movement field |

| `Rundvekt over kvote` | 1,008,689 | 93.4033 | Optional exception measure |

| `Bruttotonnasje 1969` | 952,775 | 88.2257 | Vessel attribute with low coverage |

| `Landingsfylke` | 17,721 | 1.6409 | Relevant regional quality issue |

| `Art FAO (kode)` | 150 | 0.0139 | Reference-code coverage issue |

| `Rundvekt` | 1 | 0.0001 | Main-measure handling rule required |

Sixty fields contain at least one missing value. High missingness is not automatically
an error: several fields are optional or only apply to particular document types.


## 3. Typed checks for dates and weights


In [ ]:
date_profile = []
for column in ["Siste fangstdato", "Landingsdato", "Dokument salgsdato"]:
    parsed = F.to_date(q(column), "dd.MM.yyyy")
    result = df.select(q(column).alias("raw"), parsed.alias("parsed")).agg(
        F.sum(F.when(F.trim("raw") == "", 1).otherwise(0)).alias("missing"),
        F.sum(F.when((F.trim("raw") != "") & F.col("parsed").isNull(), 1)
             .otherwise(0)).alias("invalid"),
        F.min("parsed").alias("minimum"),
        F.max("parsed").alias("maximum"),
    ).first()
    date_profile.append({
        "column": column,
        "missing_count": int(result["missing"]),
        "invalid_count": int(result["invalid"]),
        "minimum": result["minimum"].isoformat() if result["minimum"] else None,
        "maximum": result["maximum"].isoformat() if result["maximum"] else None,
    })

weight_profile = []
for column in ["Produktvekt", "Rundvekt", "Bruttovekt"]:
    parsed = F.regexp_replace(F.trim(q(column)), ",", ".").cast("decimal(24,6)")
    result = df.select(q(column).alias("raw"), parsed.alias("parsed")).agg(
        F.sum(F.when(F.trim("raw") == "", 1).otherwise(0)).alias("missing"),
        F.sum(F.when((F.trim("raw") != "") & F.col("parsed").isNull(), 1)
             .otherwise(0)).alias("invalid"),
        F.sum(F.when(F.col("parsed") < 0, 1).otherwise(0)).alias("negative"),
        F.sum(F.when(F.col("parsed") == 0, 1).otherwise(0)).alias("zero"),
        F.min("parsed").alias("minimum"),
        F.max("parsed").alias("maximum"),
    ).first()
    weight_profile.append({
        "column": column,
        "missing_count": int(result["missing"]),
        "invalid_count": int(result["invalid"]),
        "negative_count": int(result["negative"]),
        "zero_count": int(result["zero"]),
        "minimum": str(result["minimum"]) if result["minimum"] is not None else None,
        "maximum": str(result["maximum"]) if result["maximum"] is not None else None,
    })

spark.createDataFrame(date_profile).show(truncate=False)
spark.createDataFrame(weight_profile).show(truncate=False)


### Typed-check results

| Date field | Missing | Invalid | Minimum | Maximum |

|---|---:|---:|---:|---:|

| `Siste fangstdato` | 0 | 0 | 2022-12-06 | 2023-12-31 |

| `Landingsdato` | 0 | 0 | 2022-12-06 | 2024-12-30 |

| `Dokument salgsdato` | 0 | 0 | 2022-12-06 | 2025-08-20 |

| Weight field | Missing | Invalid | Negative | Zero | Minimum | Maximum |

|---|---:|---:|---:|---:|---:|---:|

| `Produktvekt` | 0 | 0 | 0 | 68,864 | 0 | 3,516,887 |

| `Rundvekt` | 1 | 0 | 0 | 169,439 | 0 | 14,731,860 |

| `Bruttovekt` | 0 | 0 | 0 | 35,591 | 0 | 3,516,887 |

The date ranges are not parsing failures. Catch, landing and document dates describe
different events and can cross calendar years. Zero weights and the high maximum values
need business rules; profiling alone does not prove they are errors.


## 4. Duplicate rows and candidate key


In [ ]:
key_columns = [
    "Fangstår", "Salgslag (kode)", "Dokumenttype (kode)",
    "Dokumentnummer", "Linjenummer",
]

complete_key = F.lit(True)
for column in key_columns:
    complete_key = complete_key & q(column).isNotNull() & (F.trim(q(column)) != "")

incomplete_key_rows = df.filter(~complete_key).count()
duplicate_key_groups = (
    df.filter(complete_key)
      .groupBy(*[q(column) for column in key_columns])
      .count()
      .filter(F.col("count") > 1)
)
key_stats = duplicate_key_groups.agg(
    F.count("*").alias("groups"),
    F.sum(F.col("count") - 1).alias("extra_rows"),
    F.max("count").alias("max_group_size"),
).first()

# SHA-256 makes the full-row comparison compact; collision risk is negligible.
row_hash = F.sha2(F.to_json(F.struct(*[q(c) for c in source_columns])), 256)
exact_duplicate_extra_rows = row_count - df.select(
    row_hash.alias("row_hash")
).distinct().count()

duplicates = {
    "exact_duplicate_extra_rows": exact_duplicate_extra_rows,
    "candidate_key": key_columns,
    "incomplete_key_rows": incomplete_key_rows,
    "duplicate_key_groups": int(key_stats["groups"] or 0),
    "duplicate_key_extra_rows": int(key_stats["extra_rows"] or 0),
    "maximum_key_group_size": int(key_stats["max_group_size"] or 0),
}
duplicates


The full snapshot has **no repeated full-row hashes**, and the candidate key
is complete and unique across all 1,079,929 rows. It is therefore a strong Silver-key
candidate, subject to agreement in A5. `Dokumentnummer` alone is not sufficient: it has
227,967 distinct raw values for more than one million product lines.


## 5. Distributions for analysis-driving fields


In [ ]:
distribution_columns = [
    "Fangstår", "Landingsmåned (kode)", "Landingsfylke",
    "Landingsnasjon", "Art", "Art - gruppe",
]
distributions = {}

for column in distribution_columns:
    rows = (
        df.groupBy(q(column)).count()
          .orderBy(F.desc("count"), q(column).asc_nulls_first())
          .limit(20 if column == "Landingsmåned (kode)" else 10)
          .collect()
    )
    distributions[column] = [
        {"value": row[column], "count": int(row["count"])} for row in rows
    ]
    print(f"\n{column}")
    for item in distributions[column]:
        print(f"  {str(item['value']):35} {item['count']:>10,}")


Distribution highlights:

- All rows have `Fangstår = 2023`.

- March is the largest landing month with 213,496 rows.

- Month code `13` means *Levert påfølgende år* (delivered the following year)
  and occurs in 1,456 rows; it is a documented category, not month 13.

- `Troms og Finnmark` is the largest landing county with 427,794 rows; the empty
  county value occurs 17,721 times.

- Norway accounts for 1,065,314 landing rows.

- Cod (`Torsk`) is the largest species with 312,385 rows.

These are row distributions, not weight totals. A document product line is the source
grain, so row counts must not be presented as catch volume.


## 6. Conclusions for A5-A7


| Finding | Count | Required follow-up |

|---|---:|---|

| Candidate-key rows missing a component | 0 | Propose the five-field key in A5 |

| Candidate-key duplicate groups | 0 | Still define deterministic future handling |

| Exact duplicate extra rows | 0 | Keep exact-deduplication as a defensive rule |

| Missing `Landingsfylke` | 17,721 | Preserve and flag; do not invent a county |

| Missing `Art FAO (kode)` | 150 | Preserve FDIR species; flag missing FAO mapping |

| Missing `Rundvekt` | 1 | Quarantine or flag according to the agreed measure |

| Zero `Rundvekt` | 169,439 | Validate meaning before treating zero as invalid |

| Invalid tested dates or weights | 0 | Use explicit parsing rules in Silver |

Recommended Silver grain: one current document product line. Recommended key:
`Fangstår`, `Salgslag (kode)`, `Dokumenttype (kode)`, `Dokumentnummer`,
`Linjenummer`. These are profiling recommendations, not final model decisions; A5
records the agreement and A6-A7 implement the rules.


## 7. Save reproducible evidence


In [ ]:
evidence = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "snapshot": str(snapshot.relative_to(ROOT)).replace("\\", "/"),
    "schema_version": expected["version"],
    "row_count": row_count,
    "source_column_count": len(source_columns),
    "field_profile": field_profile,
    "date_profile": date_profile,
    "weight_profile": weight_profile,
    "duplicates": duplicates,
    "distributions": distributions,
}

evidence_path = ROOT / "docs/evidence/a4-profile.json"
evidence_path.parent.mkdir(parents=True, exist_ok=True)
evidence_path.write_text(
    json.dumps(evidence, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved {evidence_path.relative_to(ROOT)}")
